# CardioIA – Fase 2 | Parte 1
## Extração de Sintomas e Sugestão de Diagnóstico

**Grupo 69 – FIAP**

---

Este notebook implementa um sistema básico de apoio ao diagnóstico cardiovascular.  
O código lê frases de sintomas relatados por pacientes (arquivo `.txt`), identifica expressões-chave  
com base em um mapa de conhecimento (arquivo `.csv`) e sugere possíveis diagnósticos.

**Arquivos utilizados:**
- `sintomas_frases.txt` — 10 frases simulando relatos de pacientes
- `mapa_conhecimento.csv` — 250 pares de sintomas associados a doenças cardiovasculares

## 1. Importações

In [1]:
import csv
import re

## 2. Carregar o Mapa de Conhecimento (CSV)

O arquivo `mapa_conhecimento.csv` contém 250 registros com a estrutura:  
`Sintoma1 | Sintoma2 | Doença Associada`

Cobre doenças como Infarto do Miocárdio, Insuficiência Cardíaca, AVC, Arritmia, Hipertensão, entre outras.

In [ ]:
def carregar_mapa(caminho_csv='../dataset/mapa_conhecimento.csv'):
    """
    Lê o arquivo CSV e retorna uma lista de dicionários com
    os campos: sintoma1, sintoma2, doenca_associada.
    """
    mapa = []
    with open(caminho_csv, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for linha in reader:
            mapa.append({
                'sintoma1': linha['Sintoma1'].strip().lower(),
                'sintoma2': linha['Sintoma2'].strip().lower(),
                'doenca': linha['Doenca_Associada'].strip()
            })
    return mapa

mapa = carregar_mapa()
print(f'Mapa de conhecimento carregado: {len(mapa)} registros')
print('\nExemplos de entradas:')
for entrada in mapa[:5]:
    print(f"  {entrada['sintoma1']} | {entrada['sintoma2']} -> {entrada['doenca']}")

## 3. Carregar as Frases de Sintomas (TXT)

O arquivo `sintomas_frases.txt` contém 10 frases que simulam relatos reais de pacientes,  
descrevendo o que sentem, quando os sintomas começaram e como afetam sua rotina.

In [ ]:
def carregar_frases(caminho_txt='../dataset/sintomas_frases.txt'):
    """
    Lê o arquivo TXT e retorna uma lista de frases limpas.
    Remove numeração inicial (ex: '1. ', '2. ').
    """
    with open(caminho_txt, encoding='utf-8') as f:
        linhas = f.readlines()
    frases = []
    for linha in linhas:
        linha = linha.strip()
        if linha:
            linha = re.sub(r'^\d+\.\s*', '', linha)
            frases.append(linha)
    return frases

frases = carregar_frases()
print(f'Frases carregadas: {len(frases)}\n')
for i, frase in enumerate(frases, start=1):
    print(f'Paciente {i:02d}: {frase}')
    print()

## 4. Identificar Sintomas e Sugerir Diagnósticos

A lógica compara cada frase do paciente com o mapa de conhecimento.  
Quando um sintoma é encontrado na frase, a doença associada é sugerida como diagnóstico.

In [4]:
def identificar_sintomas_e_diagnostico(frase, mapa):
    """
    Percorre o mapa de conhecimento e verifica se algum sintoma
    (sintoma1 ou sintoma2) esta contido na frase.
    Retorna uma lista de diagnosticos sugeridos.
    """
    frase_lower = frase.lower()
    resultados = []
    doencas_encontradas = set()

    for entrada in mapa:
        sintoma1 = entrada['sintoma1']
        sintoma2 = entrada['sintoma2']
        doenca   = entrada['doenca']

        achou_s1 = sintoma1 in frase_lower
        achou_s2 = sintoma2 in frase_lower

        if (achou_s1 or achou_s2) and doenca not in doencas_encontradas:
            sintomas_achados = []
            if achou_s1:
                sintomas_achados.append(sintoma1)
            if achou_s2:
                sintomas_achados.append(sintoma2)
            resultados.append({
                'sintomas_identificados': sintomas_achados,
                'diagnostico_sugerido': doenca
            })
            doencas_encontradas.add(doenca)

    return resultados

## 5. Executar a Análise — Resultado Final

In [5]:
print('=' * 60)
print('  CardioIA - Parte 1: Analise de Sintomas')
print('=' * 60)
print(f'  Mapa de conhecimento: {len(mapa)} entradas')
print(f'  Frases carregadas   : {len(frases)}')
print('=' * 60)

for i, frase in enumerate(frases, start=1):
    print(f'\nPaciente {i:02d}: {frase}')
    resultados = identificar_sintomas_e_diagnostico(frase, mapa)

    if resultados:
        print(f'  -> {len(resultados)} possivel(is) diagnostico(s):')
        for r in resultados:
            sintomas_str = ', '.join(r['sintomas_identificados'])
            print(f'     * {r["diagnostico_sugerido"]}')
            print(f'       (Sintomas detectados: {sintomas_str})')
    else:
        print('  -> Nenhum diagnostico identificado no mapa de conhecimento.')

print('\n' + '=' * 60)
print('  Analise concluida.')
print('=' * 60)

  CardioIA - Parte 1: Analise de Sintomas
  Mapa de conhecimento: 250 entradas
  Frases carregadas   : 10

Paciente 01: Há dois dias estou com uma dor no peito que piora quando faço esforço físico, como subir escadas, e isso tem me impedido de trabalhar normalmente.
  -> 3 possivel(is) diagnostico(s):
     * Infarto do Miocárdio
       (Sintomas detectados: dor no peito)
     * Trombose Venosa Profunda
       (Sintomas detectados: dor)
     * Miocardite
       (Sintomas detectados: dor no peito)

Paciente 02: Sinto cansaço constante há uma semana, mesmo depois de uma noite inteira de descanso, e tenho dificuldade para realizar tarefas simples do dia a dia.
  -> 11 possivel(is) diagnostico(s):
     * Insuficiência Cardíaca
       (Sintomas detectados: cansaço constante)
     * Angina
       (Sintomas detectados: cansaço)
     * Arritmia Cardíaca
       (Sintomas detectados: cansaço)
     * Endocardite
       (Sintomas detectados: cansaço)
     * Pericardite
       (Sintomas detectados: 

---

## Considerações sobre Governança de Dados e Viés em IA

Durante o desenvolvimento deste módulo, foram considerados princípios de **Governança de Dados** e **Viés em IA**:

**Qualidade dos dados:**
- O mapa de conhecimento foi construído com base em terminologia clínica reconhecida, cobrindo 30+ condições cardiovasculares.
- As frases de sintomas simulam relatos reais e variados, incluindo faixas etárias e contextos diferentes.

**Limitações e riscos de viés:**
- O sistema utiliza correspondência por substring (busca exata de termos), o que pode gerar falsos positivos quando palavras curtas como 'dor' aparecem em contextos diferentes.
- Frases com vocabulário coloquial ou regional podem não ser reconhecidas pelo mapa de conhecimento.
- O sistema não considera a combinação de múltiplos sintomas para refinar o diagnóstico — cada sintoma é tratado de forma independente.

**Próximos passos recomendados:**
- Ampliar o mapa de conhecimento com mais variações linguísticas.
- Implementar pontuação de confiança baseada no número de sintomas correspondentes.
- Incorporar dados demográficos (idade, sexo) para diagnósticos mais precisos.